# 11 · Why workflows come before autonomy: chaining, routing and parallel calls
Level L3 · Part 3 · core · ~25 min · builds on 06

## Goal
Compare a fixed workflow with an agent on cost, reliability and debuggability, then build the three basic workflow
shapes: a prompt chain with a gate, a router, and parallel calls. Each shape makes a known number of model calls.

## Why it matters
Lesson 06 gave the model a loop: the model picks each next step. That fits steps you cannot know in advance. Many
support tasks are known in advance. For those, an agent pays for steps you did not need, and you cannot say how many
calls a message will take.

| Without a fixed workflow (agent loop) | With a fixed workflow |
|---|---|
| The number of calls changes from one message to the next | Each message makes the same calls in the same order |
| A wrong reply shows only the final text | Each step returns its own output, so you can test it |
| You learn the cost after the run; only a cap limits it | You price a message before it runs: calls times tokens times price |

**What it makes easy:** each step is a small function you can test, log and gate. **When you don't need it:** a
question that needs one model call.

## The idea
A **workflow** is a fixed path your code writes, with a model filling in some steps. An **agent** lets the model
choose the path. Three shapes cover most fixed workflows. **Prompt chaining** runs steps in order, and a **gate**
(a plain Python check) decides whether the next step runs. **Routing** uses one call to pick a branch, and code sends
the work down that branch. **Parallel calls** run independent jobs at the same time, such as routing and extracting
from one message.

Analogy: a restaurant kitchen line. Each station does one job and a check happens before the plate moves on. An agent
is a cook who decides every step as they go. Start with the line; add the cook only where the order really needs it.

## Picture
Without a workflow, the model decides the path. With one, your code decides it and the model fills in the steps.

```mermaid
flowchart LR
  subgraph Without: the agent decides
    M1[Customer message] --> A[Model picks the next step]
    A -->|asks for a tool| T[Tool runs]
    T --> A
    A -->|decides it is done| R1[Reply]
  end
```

```mermaid
flowchart LR
  subgraph With: the workflow decides
    M2[Customer message] --> RT[Route: call 1]
    M2 --> X[Extract order number: call 2]
    RT --> G{Order gate: order_status and order exists?}
    X --> G
    G -->|yes| D[Draft reply: call 3]
    G -->|no| Q[Queue for a person]
  end
```

## Already know this?
Part 3 (lessons 11–18) adds workflows, approval, memory and retrieval to the agent loop. If you can answer all four,
skim this lesson and go on to lesson 12, which builds on the three shapes here:

1. What is the difference between a workflow and an agent, and which one makes a fixed number of model calls?
2. In a prompt chain, what does a gate decide, and what should the chain do when the gate fails?
3. Why does a router use one call to pick a branch, and what should the code do with a label it does not know?
4. Why do parallel calls cut waiting time but not cost?

## Step by step
**Step 1 · setup (one client for the whole lesson).** The key is read from `.env`, never typed here. While a
recording replays, a placeholder is used and no key is sent. No key is needed to replay the recorded calls; a key is only
needed to record new ones. `llm_replay` is a course helper, not part of the SDK: it
saves each model call the first time. `client.aio` is the SDK's async interface, so several calls can wait at once.

In [1]:
import sys; sys.path.insert(0, "..")
import asyncio
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("11_why-workflows-first")

from google import genai  # the official Google SDK
from google.genai import types
from pydantic import BaseModel
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"   # the course's pinned model, as in lessons 02 to 06
PRICE_IN, PRICE_OUT = 0.25, 1.50  # USD per 1M tokens, as checked for lesson 02 on 2026-10-20
client = genai.Client(  # in your own project keep vertexai=True and pass your own key
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx
)
print("client ready; replaying a recording:", llm_replay.is_replaying())

LLM mode: replay (11_why-workflows-first.yaml)


client ready; replaying a recording: True


**The one helper.** `ask` makes one awaited call and logs it: the step name, the tokens and the cost. Every model call
in this lesson goes through it, so Step 6 can add up the bill.

In [2]:
LOG = []  # every model call this lesson makes: (step, input tokens, output tokens, USD)


async def ask(step: str, prompt: str, config: types.GenerateContentConfig | None = None):
    """One awaited model call. It is logged so the cost can be read at the end."""
    resp = await client.aio.models.generate_content(model=MODEL, contents=prompt, config=config)
    u = resp.usage_metadata
    out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    LOG.append((step, u.prompt_token_count, out, (u.prompt_token_count * PRICE_IN + out * PRICE_OUT) / 1_000_000))
    return resp

**Step 2 · the price of a step (no call).** Before you build anything, price the two options: a fixed workflow of a
few calls, and an agent that makes between two and six calls (an assumption for the sketch), each about 1,000 input and
100 output tokens. These are worked numbers, not a measurement, and the lesson shows a known call count, not that the
workflow is cheaper. Step 6 measures the real calls.

In [3]:
def step_usd(tokens_in: int = 1000, tokens_out: int = 100) -> float:
    return (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000


print(f"one step: ${step_usd():.6f}")
for calls in (2, 6):
    print(f"{calls} model calls per message: ${calls * step_usd():.6f}")

one step: $0.000400
2 model calls per message: $0.000800
6 model calls per message: $0.002400


**Step 3 · routing and extraction at the same time (parallel calls).** Both read the same message but do not need
each other's answers, so both calls run together with `asyncio.gather`. The router answers with one
label. The route check is code: a label outside the list goes to the general queue, never to a guess.

In [4]:
ORDERS = {  # the same fictional rows as lessons 05 and 06
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}


def lookup_order(order_id: str) -> dict:
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


ROUTES = {  # each label names a queue; code decides what happens next
    "order_status": "Answer from the order record.",
    "refund": "Send to the refund queue: a person approves any money back.",
    "other": "Send to the general queue.",
}
ROUTE_CONFIG = types.GenerateContentConfig(
    temperature=0,
    system_instruction=f"Label the customer message with exactly one of: {', '.join(ROUTES)}. Reply with the label only.")


def route_check(label: str) -> str:
    """Route check: anything that is not a known label goes to the general queue."""
    label = label.strip().lower()
    return label if label in ROUTES else "other"

The extraction asks for one thing: the order number, as a JSON object. `Extract` is the shape of that answer.

In [5]:
class Extract(BaseModel):  # the extraction's answer: the order number, or null when the customer gave none
    order_id: str | None


EXTRACT_CONFIG = types.GenerateContentConfig(
    temperature=0, response_mime_type="application/json", response_json_schema=Extract.model_json_schema(),
    system_instruction="Copy the order number the customer wrote, digits only. Use null if there is none.")
print("the reply must have these fields:", list(Extract.model_json_schema()["properties"]))

the reply must have these fields: ['order_id']


Steps 3 and 4 are one workflow: calls 1 and 2 run in parallel, then the route check and the order gate, then call 3 (a chain).
This cell needs the setup and `ask` cells above it to have run first. `gather` returns the answers in the order the calls were listed.

In [6]:
MSG ="Where is order 1093? It should have arrived three days ago."
route_reply, extract_reply = await asyncio.gather(
    ask("route", MSG, ROUTE_CONFIG),  # call 1: the router
    ask("extract", MSG, EXTRACT_CONFIG),  # call 2: the chain's first step, at the same time
)
queue = route_check(route_reply.text)
print("queue:", queue, "|", ROUTES[queue])

queue: order_status | Answer from the order record.


**Step 4 · chaining with an order gate: extract, check, then draft.** Run the gather cell above first: this cell reads
`queue` and `extract_reply` from it. The draft (call 3) runs only when both checks pass: the route check chose
`order_status`, and the order gate found the order. The draft sees only the facts the order gate found. The recording
holds one message (order 1093), so this run shows the order-status branch. The closed gate and the general queue are
not shown by a model run here; Step 5 tests them in code.

In [7]:
DRAFT_CONFIG = types.GenerateContentConfig(temperature=0, max_output_tokens=200)


def order_gate(found: Extract) -> dict | None:
    """Order gate between steps: None stops the chain, and the facts are the only input the draft may use."""
    if found.order_id is None:
        return None
    facts = lookup_order(found.order_id)
    return None if "error" in facts else facts


found = Extract.model_validate_json(extract_reply.text)
facts = order_gate(found) if queue == "order_status" else None
if facts is None:
    print("order gate closed: no draft; the route check or the order gate stopped the chain")
else:
    draft = await ask("draft", f"Order facts: {facts}\nCustomer message: {MSG}\n"
                               "Write a two-sentence reply to the customer. Use only these facts.", DRAFT_CONFIG)  # call 3
    print("draft:", draft.text.strip())

draft: Your order 1093 is currently in transit. It is three days late.


**Step 5 · the checks, tested without a model call.** The route check and the order gate are plain functions, so you can
test them with values you write yourself. That tests the code, not the model.

In [8]:
print(route_check("refund"), "|", route_check(" Order_Status "), "|", route_check("pizza"))  # no model call
print("order gate on no order number:", order_gate(Extract(order_id=None)))  # no model call

refund | order_status | other
order gate on no order number: None


**Step 6 · what the workflow cost (no call).** Every call is in `LOG`, so the cost of the whole workflow is a sum.
The steps are named, so a costly step is easy to find. Compare this table with the worked numbers in Step 2.

In [9]:
for step, tin, tout, usd in LOG:
    print(f"{step:8} in {tin:>5}  out {tout:>4}  ${usd:.6f}")
print(f"total: {len(LOG)} model calls, ${sum(row[3] for row in LOG):.6f}")
llm_replay.stop()  # the last model call of this notebook

route    in    40  out    3  $0.000015
extract  in   114  out   16  $0.000053
draft    in    55  out   18  $0.000041
total: 3 model calls, $0.000108


**Reading the numbers.** The real calls were smaller than the worked example in Step 2: about 40 to 115 input tokens
each, not 1,000. So the worked price overstates this message. Price a real workflow from its own logged calls, and
keep the worked numbers for a rough budget before you have data. Three calls cost about $0.0001, a hundredth of a cent.

## How it simplifies the architecture
| Before: one agent loop | After: three named shapes |
|---|---|
| One loop decides the path; the caps guard the cost | Each step is a function; the code decides the path |
| A failure shows up as a wrong final reply | A failure shows up at one step, with its input and output |
| The number of calls changes per message | The number of calls is fixed per shape and can be priced |
| Adding a check means changing the prompt and hoping | Adding a check is one more gated step with its own test |

**What this does not solve:** a fixed path cannot handle a message whose next step you cannot predict, and an error
in an early step still flows into the later ones. **Alternatives:** an agent loop (lesson 06) for open tasks, and a
graph with state (lesson 19) when the workflow needs loops and pauses.

## At work
- Teams start with the simplest fixed workflow and add an agent only where the next step depends on what the model
  finds. Anthropic's guide "Building effective agents" (2024-12-19) makes this its first rule → lesson 12 (the next
  shape up: an orchestrator that splits the work).
- An agent loop costs about 4 times the tokens of one chat turn, and a multi-agent system about 15 times, per
  Anthropic's figures. Teams price a workflow before they ship it and send the easy labels to a cheaper model
  → lesson 33 (cost caps and routing).
- Parallel calls cut waiting time, not cost: you pay for every branch, and one slow branch sets the wait. Each branch
  needs its own timeout and retry → lesson 32 (reliability). LangGraph draws these shapes as graphs in lesson 19.

## Common mistakes
1. **Calling `asyncio.run` in a notebook.** A notebook already runs an event loop, so `asyncio.run` refuses to start a
   second one. Use `await` in the cell, as this lesson does. (this cell raises on purpose)

In [10]:
async def no_call():
    return "no model involved"


asyncio.run(no_call())  # inside a notebook the loop is already running

RuntimeError: asyncio.run() cannot be called from a running event loop

2. **Gathering without `return_exceptions=True`.** One failed job then raises, and the answers of the other jobs are
   lost. The cell below raises on purpose (this cell raises on purpose), with no model call.

In [11]:
async def bad_job():
    raise ValueError("the router's reply was not valid JSON")


async def good_job():
    return "order_status"


await asyncio.gather(bad_job(), good_job())  # the good answer is lost with the error

ValueError: the router's reply was not valid JSON

The fix is `return_exceptions=True`, as in the Try it below. Each failed job then comes back as an exception object, so
the code can count it as a fail and the other results still come back.

3. **Adding an agent before the workflow works.** A workflow with one wrong gate is easy to fix. An agent with a wrong
   tool loses the trace of why it went wrong.

## Under the hood
`asyncio.gather` starts every coroutine, waits for all of them and returns the results in the order you passed them,
not the order they finished. This sketch uses a sleep for the network wait. No model is called here.

In [12]:
async def slow(name: str, secs: float) -> str:
    await asyncio.sleep(secs)  # stands in for the network wait of one model call
    return name


loop = asyncio.get_running_loop()
t0 = loop.time()
done = await asyncio.gather(slow("route", 0.3), slow("extract", 0.2))
print(done, f"in {loop.time() - t0:.1f} s, not 0.5 s: the waits overlap")

['route', 'extract'] in 0.3 s, not 0.5 s: the waits overlap


## Try it
Write `failed_jobs(names, results)` that takes the names of the jobs and the list from `gather(...,
return_exceptions=True)`, and returns the names of the jobs whose result is an exception. No model call is needed.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
def failed_jobs(names, results) -> list[str]:
    return [n for n, r in zip(names, results) if isinstance(r, BaseException)]


print(failed_jobs(["route", "extract"], ["refund", "1093"]))                  # []
print(failed_jobs(["route", "extract"], ["refund", ValueError("not JSON")]))  # ['extract']
print(failed_jobs(["route", "extract"], [TimeoutError("slow"), "1093"]))      # ['route']

[]
['extract']
['route']


## Recap
- A workflow is a fixed path your code writes; an agent lets the model choose the path. Workflows have known costs.
- Chaining runs steps in order with a gate between them; an order gate that fails stops the chain before the next call.
- Routing uses one call to choose a branch; anything outside the known labels goes to a safe default.
- Parallel calls run independent jobs at the same time with `asyncio.gather`; `return_exceptions=True` keeps one
  failure from hiding the rest.

**Interview:** "When would you build a workflow instead of an agent?" Strong answer: "When I can write the steps in
advance. The workflow has a known number of calls, each step is testable, and a failure shows at one step. I add an
agent only at the step where the next action depends on what the model finds."